In [1]:
import numpy as np

from scipy.optimize import NonlinearConstraint
from scipy.optimize import minimize

In [2]:
def unit_vector(v):
    return v / np.linalg.norm(v)

def spherique_vers_cartesien(r, theta, phi):
    x = r * np.sin(theta) * np.cos(phi)
    y = r * np.sin(theta) * np.sin(phi)
    z = r * np.cos(theta)
    return np.array([x, y, z], dtype=float)

In [3]:
def force_moment(fi, Pi, Qi):

    u = unit_vector(Qi - Pi)

    F = fi * u

    M = np.cross(Pi, F)

    return F, M

In [9]:
def Residu(x, Q, r):

    n = len(Q)

    f = x[:n]
    theta = x[n:2*n]
    phi = x[2*n:3*n]

    Ftot = np.zeros(3)
    Mtot = np.zeros(3)

    for i in range(n):

        Pi = spherique_vers_cartesien(r, theta[i], phi[i])

        F, M = force_moment(f[i], Pi, Q[i])

        Ftot += F
        Mtot += M

    return np.concatenate((Ftot, Mtot))

In [ ]:
def J(x, Q, r, R, f0, alpha_f, alpha_P): # important de déclarer x en premier pour minimize

    n = len(Q)

    f = x[:n]
    theta = x[n:2*n]
    phi = x[2*n:3*n]

    phi_f = np.sum((f - f0)**2)

    phi_P = 0.0

    for i in range(n):

        Pi = spherique_vers_cartesien(r, theta[i], phi[i])

        Pstar = (r/R) * Q[i]

        phi_P += np.sum((Pi - Pstar)**2)

    return (alpha_f * phi_f + alpha_P * phi_P)

In [ ]:
beams = {}

with open("../facility_config_files/xavier_ico30_theta_phi_rad.txt", "r") as f:
    for i, line in enumerate(f):
        line = line.strip() # enlève les espaces, tabulations et retours à la ligne au début et à la fin de la chaîne

        # Ignore les lignes vides
        if not line:
            continue # passse à l'itération suivante

        elements = line.split() # on split la ligne à l'espace
        theta = float(elements[0])
        phi = float(elements[1])

        beams[i] = {"theta": theta,"phi": phi}

n = len(beams)
r = 1.94e-3 #m
R = 10.0 #m
Q = []
for i in range(n):
    Q.append(spherique_vers_cartesien(R,beams[i]['theta'],beams[i]['phi']))

P_theta = np.array([beams[i]["theta"] for i in range(n)])

P_phi = np.array([beams[i]["phi"] for i in range(n)])

f = np.ones(n)
f[0] = 0.9

x0 = np.concatenate([f, P_theta, P_phi])

In [ ]:
# Contrainte d'équilibre : 0 <= R(x) <= 0  donc R(x) = 0
eq_constraint = NonlinearConstraint(lambda x: Residu(x, Q, r),lb=np.zeros(6),ub=np.zeros(6)) # (contrainte(x), lower_bound, upper_bound) 

In [ ]:
f0 = 1.0
alpha_f = 1.0
alpha_P = 1.0
arguments = (Q, r, R, f0, alpha_f, alpha_P)

res = minimize(J, x0, args = arguments, constraints = [eq_constraint], method="trust-constr") # J(x0,args)

print(f"L'algorithme a convergé : {res.success}.")
print(f"La valeur atteinte de la fonction J est : {res.fun}.")
print(res.message)

L'algorithme a convergé : True, la valeur atteinte de la fonction objectif est : 1.7554320100047945e-15.
`gtol` termination condition is satisfied.


In [16]:
res.x

array([ 9.99999990e-01,  9.99999990e-01,  9.99999990e-01,  9.99999990e-01,
        9.99999991e-01,  9.99999992e-01,  9.99999992e-01,  9.99999993e-01,
        9.99999992e-01,  9.99999992e-01,  9.99999991e-01,  9.99999990e-01,
        9.99999992e-01,  9.99999991e-01,  9.99999991e-01,  9.99999993e-01,
        9.99999993e-01,  9.99999994e-01,  9.99999995e-01,  9.99999995e-01,
        9.99999995e-01,  9.99999995e-01,  9.99999994e-01,  9.99999992e-01,
        9.99999993e-01,  9.99999993e-01,  9.99999995e-01,  9.99999995e-01,
        9.99999995e-01,  9.99999993e-01,  3.74317755e-13,  6.28318530e-01,
        6.28318530e-01,  1.25663706e+00,  1.04719755e+00,  1.57079633e+00,
        1.04719755e+00,  1.25663706e+00,  6.28318530e-01,  6.28318530e-01,
        1.04719755e+00,  1.25663706e+00,  1.88495559e+00,  1.57079633e+00,
        1.88495559e+00,  2.51327412e+00,  2.09439510e+00,  2.09439510e+00,
        2.51327412e+00,  1.88495559e+00,  1.57079633e+00,  1.88495559e+00,
        1.25663706e+00,  